In [ ]:
import pandas as pd
import os
from model_metrics import (
    combine_plots,
    show_roc_curve,
    show_pr_curve,
    show_calibration_curve,
    summarize_model_performance,
)

from folktables import ACSDataSource, ACSIncome

In [ ]:
ds = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
ca = ds.get_data(states=["CA"], download=True)
X_acs, y_acs, group = ACSIncome.df_to_pandas(ca)

In [ ]:
y_acs.value_counts()

In [ ]:
data_path = "../model_files/"

In [ ]:
X_test = pd.read_parquet(os.path.join(data_path, "X_test.parquet"))
y_test = pd.read_parquet(os.path.join(data_path, "y_test.parquet"))

In [ ]:
from model_metrics.model_registry  import available, load_all, load_model

available()  
from model_metrics.model_registry import best_per_algo, load_best_per_algo

best_per_algo(metric="valid AUC ROC")
champs = load_best_per_algo(metric="valid AUC ROC")
champs
model_rf = champs["rf_income"]
model_dt = champs["dt_income"]
model_lr = champs["lr_income"]

In [ ]:
model_rf.get_feature_names()

In [ ]:
from model_metrics import align_features

ext = align_features(
    X_acs,
    model=model_rf,
    col_map={
        "AGEP": "age",
        "SCHL": "education-num",
        "WKHP": "hours-per-week",
    },
    copy_passthrough=True,
    on_unmapped="warn",
)
p_acs = model_rf.predict_proba(ext)[:, 1]

In [ ]:
threshold = next(iter(model_rf.threshold.values()))    
threshold

In [ ]:
import numpy as np
from model_metrics import summarize_model_performance

summarize_model_performance(
    y_prob=p_acs,
    y=np.asarray(y_acs["PINCP"]).astype(int).ravel(),
    model_title="RF (ACS CA 2018)",
    return_df=True,
    decimal_places=3,
    model_threshold=threshold,
)

In [ ]:
summarize_model_performance(
    y_prob=p_acs,
    y=np.asarray(y_acs["PINCP"]).astype(int).ravel(),
    model_title="RF (ACS CA 2018)",
    return_df=True,
    decimal_places=3,
    group_category=group["RAC1P"],
)

In [ ]:
p_adult_test = model_rf.predict_proba(X_test)[:, 1]
y_adult_test = np.asarray(y_test).astype(int).ravel()

In [ ]:
import numpy as np
from model_metrics import (
    combine_plots,
    show_roc_curve,
    show_pr_curve,
    show_calibration_curve,
    plot_threshold_metrics,
)

p_acs = model_rf.predict_proba(ext)[:, 1]
y_acs_int = np.asarray(y_acs["PINCP"]).astype(int).ravel()

p_adult_test = model_rf.predict_proba(X_test)[:, 1]
y_adult_test = np.asarray(y_test).astype(int).ravel()

INT_TITLE = "Adult 1994 (internal)"
EXT_TITLE = "ACS CA 2018 (external)"

probs = [p_adult_test, p_acs]
truths = [y_adult_test, y_acs_int]
titles = [INT_TITLE, EXT_TITLE]

styles = {
    INT_TITLE: {"color": "black", "linewidth": 1.5},
    EXT_TITLE: {"color": "#C1440E", "linewidth": 1.5},
}

SHARED = {
    "y_prob": probs,
    "y": truths,
    "model_title": titles,
    "overlay": True,
    "curve_kwgs": styles,
}

combine_plots(
    plot_calls=[
        (
            show_roc_curve,
            {
                **SHARED,
                "decimal_places": 3,
                "linestyle_kwgs": {
                    "color": "gray",
                    "linestyle": "--",
                },
                "title": "Discrimination",
            },
        ),
        (
            show_pr_curve,
            {
                **SHARED,
                "decimal_places": 3,
                "legend_metric": "ap",
                "title": "Precision-Recall",
            },
        ),
        (
            show_calibration_curve,
            {
                **SHARED,
                "bins": 10,
                "brier_decimals": 3,
                "title": "Calibration",
                "legend_loc": "bottom",
            },
        ),
(
    plot_threshold_metrics,
    {
        "y_prob": probs,
        "y_test": truths,
        "model_title": titles,
        "overlay": True,
        "model_threshold": threshold,
        "title": "Threshold behavior",
        "legend_ncol": 2,
        "legend_loc": "bottom",
        "legend_bbox_to_anchor": (0.5, -0.18),
    },
),
    ],
    n_cols=2,
    n_rows=2,
    figsize=(12, 10),
    suptitle="RF income classifier: internal vs external validation",
    image_filename="../model_files/images/svg_images/ext_val_combo.svg",
)
